# 01 · Data Preparation

Audit and cleaning of the seven public IMDb tables. The raw data stays in `data/raw` and a completed preparation is published as Parquet. The profiles cover every column, before and after cleaning.

[IMDb schema](https://data.imdb.com/non-commercial-datasets/)

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next((path for path in (cwd, *cwd.parents) if (path / "src/imdb_loader.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run the notebook from the IMDb repository or its notebooks folder.")
sys.path.insert(0, str(ROOT))

from pyspark.sql import functions as F
from src.data_quality import prepare
from src.imdb_loader import data_root, spark_session

DATA = data_root(ROOT)
spark = spark_session("IMDb Data Preparation")

## Q1 · Load and measure missing values

Empty strings, whitespace-only strings and the IMDb marker are counted together with null values. The rate uses every row of the table, including those with an invalid key.

In [ ]:
snapshot = prepare(spark, DATA, expand_gzip=os.getenv("IMDB_EXPAND_GZIP", "1") != "0")
print(f"Published preparation: {snapshot}")

missing = spark.read.parquet((snapshot / "quality/missing").as_posix())
raw_profile = missing.filter(F.col("stage") == "raw")
raw_profile.orderBy("table", "column").drop("stage").withColumn(
    "missing_pct", F.round("missing_pct", 3)
).show(100, truncate=False)

## Q2 · Five most incomplete columns

The ranking compares raw percentages across all tables. Ties are broken by table name, then by column name.

In [ ]:
top_missing = raw_profile.drop("stage").orderBy(
    F.desc_nulls_last("missing_pct"), "table", "column"
).limit(5)
top_missing.show(truncate=False)

## Q3 · Anomalies and examples

The checks cover numeric types and bounds, inverted years, suspicious lifespans, invalid keys, duplicates and orphan references. A single row can fall under several checks. Orphan references are audited on the usable keys after structural cleaning.

In [ ]:
anomalies = spark.read.parquet((snapshot / "quality/anomalies").as_posix())
anomalies.select("table", "issue", "count", "unit", "action").orderBy(
    F.desc("count"), "table", "issue"
).show(100, truncate=False)
anomalies.filter(F.col("count") > 0).select("table", "issue", "example").orderBy(
    "table", "issue"
).show(100, truncate=160)

## Q4 · Treatment and before/after check

Invalid numeric fields are set to null. Rows that are identical after normalization are merged. Every variant of a conflicting key is discarded and the audit keeps two representative variants. A suspicious lifespan and orphan references remain visible for review.

In [ ]:
before = raw_profile.select("table", "column", F.col("rows").alias("raw_rows"),
                            F.col("missing").alias("raw_missing"))
after = missing.filter(F.col("stage") == "clean").select(
    "table", "column", F.col("rows").alias("clean_rows"), F.col("missing").alias("clean_missing")
)
comparison = before.join(after, ["table", "column"])
comparison.orderBy("table", "column").show(100, truncate=False)

removed = anomalies.filter(F.col("issue").isin(
    "missing_or_invalid_key", "duplicate_rows", "conflicting_rows"
)).groupBy("table").agg(F.sum("count").alias("removed"))
row_check = comparison.select("table", "raw_rows", "clean_rows").distinct().join(removed, "table")
assert row_check.filter(F.col("raw_rows") - F.col("clean_rows") != F.col("removed")).count() == 0
assert missing.filter((F.col("missing") < 0) | (F.col("missing") > F.col("rows"))).count() == 0
row_check.orderBy("table").show(truncate=False)

In [ ]:
spark.stop()